# STEP 3.0 统计建模与稳健性分析
主分析为时间分层病例交叉模型；伤亡严重度模型与网格日率比较为次级分析，只有在对应暴露表完整时才执行。

In [1]:
from pathlib import Path # 导入跨平台路径工具
import sys # 导入解释器路径模块
import pandas as pd # 导入表格处理模块
ROOT=Path.cwd().resolve() # 读取当前工作目录
ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT # 从notebooks目录回到项目根目录
sys.path.insert(0,str(ROOT)) # 将项目根目录加入模块搜索路径
from src.analysis import conditional_heat_model,heat_definition_sensitivity,event_window_sensitivity,casualty_severity_model,source_sensitivity,compare_grid_day_rates,reporting_completeness_model,consequence_association_model,fire_service_capacity_sensitivity # 导入统计建模、热浪敏感性、报告完整性、人员后果与消防能力函数
EVENTS=ROOT/'data/processed/events_enriched.csv' # 指定最终增强事件表
CASE_CONTROL=ROOT/'data/processed/case_control_weather.csv' # 指定GEE病例交叉气象暴露表
POWER_CASE_CONTROL=ROOT/'data/processed/case_control_weather_nasa_power_v10.csv' # 指定NASA POWER v10独立气象敏感性表
GRID_DAYS=ROOT/'data/processed/populated_grid_days.csv' # 指定可选网格日暴露表

In [2]:
events=pd.read_csv(EVENTS,encoding='utf-8-sig',parse_dates=['event_date']) # 读取最终增强事件表
sensitivity=source_sensitivity(events) # 计算预设队列敏感性摘要
sensitivity.to_csv(ROOT/'outputs/tables/source_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存敏感性摘要表
severity_required=['deaths','injuries','heatwave_score_0_100','population_count_2020_1km_cell','ghsl_built_volume_m3_1km','event_year'] # 定义伤亡与精简调整集均完整的严重度模型字段
severity_n=int(events.loc[events['analysis_core'],severity_required].dropna().shape[0]) # 统计严重度模型完整核心事件数
if severity_n>=30: severity_fit,severity_table,severity_sample=casualty_severity_model(events) # 在样本足够时拟合探索性伤亡严重度模型
if severity_n>=30: severity_table.to_csv(ROOT/'outputs/tables/casualty_severity_model.csv',index=False,encoding='utf-8-sig') # 保存探索性严重度模型结果
print(sensitivity,'severity_complete_cases=',severity_n) # 显示敏感性摘要与严重度完整案例数

                 cohort  n_events  n_metric      mean  median  q25  q75
0          all_geocoded       233       233  6.635419     0.0  0.0  0.0
1  extended_no_external       228       228  6.780933     0.0  0.0  0.0
2         core_verified       191       191  7.309051     0.0  0.0  0.0
3     exclude_2025_2026       111       111  8.131342     0.0  0.0  0.0
4  exclude_construction       175       175  7.200503     0.0  0.0  0.0
5         exclude_arson       185       185  7.127739     0.0  0.0  0.0 severity_complete_cases= 66


In [3]:
if CASE_CONTROL.exists(): case_control=pd.read_csv(CASE_CONTROL,encoding='utf-8-sig',parse_dates=['date']) # 在暴露表存在时读取病例交叉数据
if CASE_CONTROL.exists(): heat_fit,heat_table,heat_sample=conditional_heat_model(case_control) # 拟合时间分层病例交叉模型
if CASE_CONTROL.exists(): heat_table.to_csv(ROOT/'outputs/tables/conditional_heat_model.csv',index=False,encoding='utf-8-sig') # 保存病例交叉模型结果
if CASE_CONTROL.exists(): heat_sensitivity=heat_definition_sensitivity(case_control) # 拟合预设多阈值与持续时间敏感性模型
if CASE_CONTROL.exists(): heat_sensitivity.to_csv(ROOT/'outputs/tables/heat_definition_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存热浪定义敏感性结果
if CASE_CONTROL.exists(): era5_window_sensitivity=event_window_sensitivity(case_control,events) # 使用主ERA5产品拟合来源时期事件排除与逐洲留一分析
if CASE_CONTROL.exists(): era5_window_sensitivity.assign(weather_product='ERA5-Land').to_csv(ROOT/'outputs/tables/era5_event_window_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存主ERA5产品队列与地理稳健性结果
if CASE_CONTROL.exists(): coastal_strata=case_control.groupby('stratum_id')['era5_sampling_radius_km'].max().loc[lambda values:values.gt(0)].index # 识别采用二十公里沿海空间均值回退的病例交叉分层
if CASE_CONTROL.exists(): inland_case_control=case_control.loc[~case_control['stratum_id'].isin(coastal_strata)].copy() # 排除沿海回退分层构造像元直接采样敏感性队列
if CASE_CONTROL.exists(): inland_heat_fit,inland_heat_table,inland_heat_sample=conditional_heat_model(inland_case_control) # 拟合排除沿海回退分层的主定义条件Logit模型
if CASE_CONTROL.exists(): inland_heat_table.assign(weather_product='ERA5-Land',restriction='Exclude 20-km coastal fallback',n_observations=len(inland_heat_sample),n_strata=inland_heat_sample['stratum_id'].nunique(),n_informative_strata=inland_heat_sample.groupby('stratum_id')['heatwave_indicator'].nunique().gt(1).sum()).to_csv(ROOT/'outputs/tables/era5_coastal_exclusion_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存沿海回退排除敏感性效应量与信息量
if POWER_CASE_CONTROL.exists(): power_case_control=pd.read_csv(POWER_CASE_CONTROL,encoding='utf-8-sig',parse_dates=['date']) # 读取NASA POWER病例对照气象敏感性表
if POWER_CASE_CONTROL.exists(): power_case_control=power_case_control.loc[power_case_control['weather_status'].eq('completed')].copy() # 仅保留成功记录并显式避免接口成功定义样本
if POWER_CASE_CONTROL.exists(): power_heat_fit,power_heat_table,power_heat_sample=conditional_heat_model(power_case_control) # 拟合独立MERRA-2气象产品病例交叉模型
if POWER_CASE_CONTROL.exists(): power_heat_table.assign(weather_product='NASA POWER MERRA-2/GEOS-IT').to_csv(ROOT/'outputs/tables/nasa_power_conditional_heat_model.csv',index=False,encoding='utf-8-sig') # 保存独立气象产品主定义结果
if POWER_CASE_CONTROL.exists(): power_heat_sensitivity=heat_definition_sensitivity(power_case_control) # 拟合独立气象产品多定义敏感性分析
if POWER_CASE_CONTROL.exists(): power_heat_sensitivity.assign(weather_product='NASA POWER MERRA-2/GEOS-IT').to_csv(ROOT/'outputs/tables/nasa_power_heat_definition_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存独立气象产品热浪定义结果
if POWER_CASE_CONTROL.exists(): power_window_sensitivity=event_window_sensitivity(power_case_control,events) # 拟合独立气象产品来源时期事件排除与逐洲留一分析
if POWER_CASE_CONTROL.exists(): power_window_sensitivity.assign(weather_product='NASA POWER MERRA-2/GEOS-IT').to_csv(ROOT/'outputs/tables/nasa_power_event_window_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存独立气象产品队列与地理稳健性结果
if GRID_DAYS.exists(): rate_table=compare_grid_day_rates(pd.read_csv(GRID_DAYS,encoding='utf-8-sig')) # 在真实网格日表存在时重算火灾率
if GRID_DAYS.exists(): rate_table.to_csv(ROOT/'outputs/tables/grid_day_rate_comparison.csv',index=False,encoding='utf-8-sig') # 保存网格日率比较结果
injury_reporting_fit,injury_reporting_table,injury_reporting_sample=reporting_completeness_model(events,'injuries') # 拟合伤者人数是否报告的观察过程模型
injury_reporting_table.to_csv(ROOT/'outputs/tables/injury_reporting_completeness_model.csv',index=False,encoding='utf-8-sig') # 保存伤者报告完整性模型结果
evacuation_reporting_fit,evacuation_reporting_table,evacuation_reporting_sample=reporting_completeness_model(events,'evacuated') # 拟合疏散人数是否报告的观察过程模型
evacuation_reporting_table.to_csv(ROOT/'outputs/tables/evacuation_reporting_completeness_model.csv',index=False,encoding='utf-8-sig') # 保存疏散报告完整性模型结果
death_consequence_fit,death_consequence_table,death_consequence_sample=consequence_association_model(events,'deaths') # 拟合死亡人数与建筑及事件属性的探索关联模型
death_consequence_table.to_csv(ROOT/'outputs/tables/death_consequence_association_model.csv',index=False,encoding='utf-8-sig') # 保存死亡人数探索关联模型结果
injury_consequence_fit,injury_consequence_table,injury_consequence_sample=consequence_association_model(events,'injuries') # 拟合受伤人数与建筑及事件属性的报告概率加权探索模型
injury_consequence_table.to_csv(ROOT/'outputs/tables/injury_consequence_association_model.csv',index=False,encoding='utf-8-sig') # 保存受伤人数探索关联模型结果
capacity_sensitivity=fire_service_capacity_sensitivity(events) # 拟合CTIF四项国家消防服务能力与两类报告后果敏感性模型
capacity_sensitivity.to_csv(ROOT/'outputs/tables/ctif_fire_service_capacity_sensitivity.csv',index=False,encoding='utf-8-sig') # 保存CTIF消防服务能力敏感性结果
print('case_control_ready=',CASE_CONTROL.exists(),'power_case_control_ready=',POWER_CASE_CONTROL.exists(),'grid_days_ready=',GRID_DAYS.exists()) # 显示主分析、独立气象敏感性与率比较数据状态

case_control_ready= True power_case_control_ready= True grid_days_ready= False
